In [1]:
import pandas as pd

def make_lag_block_table(data: pd.DataFrame, T: int, d: int) -> pd.DataFrame:
    """
    参数
    ----
    data : DataFrame，含有 'index' 列为日期（如 20210104），其它列为股票代码
    T    : 与当前日的“间隔天数”
    d    : 连续取 d 天（生成 day_T+1 ... day_T+d 列）

    返回
    ----
    MultiIndex=(date, Code) 的表格，列为 day_T+1 ... day_T+d，且 date 为8位字符串（YYYYMMDD）
    """
    panel = data.copy()

    # 1) 把 index 列标准化为“日期”（内部用来排序/偏移），稍后再转回 8 位字符串
    try:
        panel["index"] = pd.to_datetime(panel["index"].astype(str), format="%Y%m%d")
    except Exception:
        panel["index"] = pd.to_datetime(panel["index"], errors="coerce")

    # 2) 宽转长
    long = (
        panel.melt(id_vars=["index"], var_name="Code", value_name="value")
          .rename(columns={"index": "date"})
          .sort_values(["Code", "date"])
    )

    # 3) 按股票分组生成 day_T+1 ... day_T+d
    g = long.groupby("Code", sort=False)["value"]
    for k in range(1, d + 1):
        offset = T + k
        col = f"day_T+{k}"
        if offset > 0:
            long[col] = g.shift(offset)
        elif offset == 0:
            long[col] = long["value"]
        else:
            long[col] = pd.NA

    # 4) 将 date 转回 8 位字符串，并返回以 (date, Code) 为 MultiIndex 的表格
    if not pd.api.types.is_datetime64_any_dtype(long["date"]):
        long["date"] = pd.to_datetime(long["date"], errors="coerce")
    long["date"] = long["date"].dt.strftime("%Y%m%d")

    out_cols = [f"day_T+{k}" for k in range(1, d + 1)]
    out = (
        long.set_index(["date", "Code"])[out_cols]
            .sort_index()
    )
    return out


In [2]:
import pandas as pd
import os

T = 5  # 间隔天数
d = 10 # 连续取 d 天
data = pd.read_feather("/home/user96/label_data1/can_trade_amt1.fea")
panel1 = make_lag_block_table(data, T=T, d=d)

data = pd.read_feather("/home/user96/label_data1/label1.fea")
panel2 = make_lag_block_table(data, T=T, d=d)

panel = panel1.join(panel2, how='inner', lsuffix='_amt', rsuffix='_label')

panel = panel.reset_index()


import numpy as np
# 1. 将 date 转换为 datetime 类型
panel["date"] = pd.to_datetime(panel["date"].astype(str), format="%Y%m%d")

# 2. 提取年份
panel["year"] = panel["date"].dt.year

# 3. 提取季度、月份、周
panel["quarter"] = panel["date"].dt.quarter
panel["month"] = panel["date"].dt.month
panel["week"] = panel["date"].dt.isocalendar().week.astype(int)

# 4. 用 sin 和 cos 做周期特征编码
# 月份 (12个月周期)
panel["month_sin"] = np.sin(2 * np.pi * panel["month"] / 12)
panel["month_cos"] = np.cos(2 * np.pi * panel["month"] / 12)

# 季度 (4个季度周期)
panel["quarter_sin"] = np.sin(2 * np.pi * panel["quarter"] / 4)
panel["quarter_cos"] = np.cos(2 * np.pi * panel["quarter"] / 4)

# 周 (52周周期)
panel["week_sin"] = np.sin(2 * np.pi * panel["week"] / 52)
panel["week_cos"] = np.cos(2 * np.pi * panel["week"] / 52)

panel.drop(columns=["quarter", "month", "week"], inplace=True)

# 将 date 转回 8 位字符串
panel["date"] = panel["date"].dt.strftime("%Y%m%d")


from sklearn.preprocessing import StandardScaler
cols_to_scale = panel.columns.difference(["date", "Code", "month_sin", "month_cos", "quarter_sin", "quarter_cos", "week_sin", "week_cos"])
scaler = StandardScaler()
panel[cols_to_scale] = scaler.fit_transform(panel[cols_to_scale])
panel = panel.reset_index()
panel = panel.fillna(0)

panel.drop(columns=["index"], inplace=True)
panel.to_feather(f"/home/user96/label_data1/hist_time.fea")

print("Data processing completed and saved to hist_data directory.")


Data processing completed and saved to hist_data directory.


In [3]:
panel

,date,Code,day_T+1_amt,day_T+2_amt,day_T+3_amt,day_T+4_amt,day_T+5_amt,day_T+6_amt,day_T+7_amt,day_T+8_amt,...,day_T+8_label,day_T+9_label,day_T+10_label,year,month_sin,month_cos,quarter_sin,quarter_cos,week_sin,week_cos
0,20170103,000001,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-1.524798,5.000000e-01,0.866025,1.000000e+00,6.123234e-17,0.120537,0.992709
1,20170103,000002,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-1.524798,5.000000e-01,0.866025,1.000000e+00,6.123234e-17,0.120537,0.992709
2,20170103,000004,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-1.524798,5.000000e-01,0.866025,1.000000e+00,6.123234e-17,0.120537,0.992709
3,20170103,000005,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-1.524798,5.000000e-01,0.866025,1.000000e+00,6.123234e-17,0.120537,0.992709
4,20170103,000006,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-1.524798,5.000000e-01,0.866025,1.000000e+00,6.123234e-17,0.120537,0.992709
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10431962,20241231,688799,-0.345952,-0.345880,-0.261339,-0.350076,-0.340292,-0.339892,-0.349970,-0.317267,...,0.252702,0.173731,-0.650571,1.530414,-2.449294e-16,1.000000,-2.449294e-16,1.000000e+00,0.120537,0.992709
10431963,20241231,688800,0.792668,0.734292,0.385844,0.695896,5.309447,6.753030,0.692366,0.542525,...,0.822351,1.298851,0.233739,1.530414,-2.449294e-16,1.000000,-2.449294e-16,1.000000e+00,0.120537,0.992709
10431964,20241231,688819,-0.336280,-0.346301,-0.307260,-0.313302,-0.344577,-0.317348,-0.199674,-0.306635,...,-0.375391,-0.206295,-0.882180,1.530414,-2.449294e-16,1.000000,-2.449294e-16,1.000000e+00,0.120537,0.992709
10431965,20241231,688981,8.311086,6.452509,8.080392,4.295832,2.729125,5.615363,3.411596,1.434700,...,-1.077751,-0.509764,-1.892715,1.530414,-2.449294e-16,1.000000,-2.449294e-16,1.000000e+00,0.120537,0.992709
